# Pipeline Financeiro 2026 — Versão Final Profissional

Notebook único para preparar, validar e exportar o dataset financeiro de 2026.

## Princípios

- dados `raw` nunca são alterados;
- cada regra de negócio é aplicada uma única vez;
- `CODACT 13` é excluído do universo financeiro;
- KILOSPO é calculado integralmente, mas apenas o que associa ao `inform_27` entra no financeiro;
- diferenças Danone são classificadas, não forçadas;
- Setembro permanece no dataset, mas é excluído da reconciliação Danone enquanto estiver incompleto;
- ocupação é calculada ao nível da viagem (`CODEUT`);
- divisões por zero devolvem `NaN`;
- a exportação só ocorre depois das validações finais.

## 00. Configuração

In [1]:
# ==========================
# 1. Imports
# ==========================
from pathlib import Path
import platform
import warnings

import duckdb
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# ==========================
# 2. Parâmetros
# ==========================
ANO = 2026
MES_VALIDACAO_DANONE_MAX = 8

CUSTO_ESTRUTURA_MENSAL = 103_000.0
EXTRA_DANONE_MENSAL = 20_000.0
DESFASAMENTO_MES_COMBUSTIVEL = -1

CLIENTES_ESPANHA = [338, 211, 15]
CODACT_EXCLUIR = [13]

TOLERANCIA_FINANCEIRA = 0.01

MAPA_CAPACIDADE = {
    4: 6,
    5: 6,
    6: 6,
    8: 12,
    12: 12,
    14: 20,
    15: 20,
    16: 20,
    18: 20,
    20: 20,
    22: 24,
    24: 24,
    33: 33,
    55: 55,
    66: 66,
}

# ==========================
# 3. Caminhos
# ==========================
def get_paths():
    sistema = platform.system()

    if sistema == "Windows":
        base = Path(r"C:\Users\LISARR\Documents\python\01.Financeiro")
        return {
            "db": Path(r"C:\Users\LISARR\OneDrive - Salvesen Logística S.A\python\00.DB\2026.duckdb"),
            "excel": base / "Danone_Custos_V3.xlsx",
            "parquet": base / "inform_27_2026_final.parquet",
            "viagens": base / "viagens_2026.parquet",
            "validacao": base / "validacao_financeira_2026.csv",
            "danone_diag": base / "validacao_danone_2026.csv",
        }

    if sistema == "Darwin":
        base = Path(
            "/Users/rr/Library/Mobile Documents/com~apple~CloudDocs/05.Salvesen"
        )
        return {
            "db": base / "00_DB" / "2026.duckdb",
            "excel": base / "Danone_Custos_v3.xlsx",
            "parquet": base / "inform_27_2026_final.parquet",
            "viagens": base / "viagens_2026.parquet",
            "validacao": base / "validacao_financeira_2026.csv",
            "danone_diag": base / "validacao_danone_2026.csv",
        }

    base = Path.cwd()
    return {
        "db": base / "2026.duckdb",
        "excel": base / "Danone_Custos_v3.xlsx",
        "parquet": base / "inform_27_2026_final.parquet",
        "viagens": base / "viagens_2026.parquet",
        "validacao": base / "validacao_financeira_2026.csv",
        "danone_diag": base / "validacao_danone_2026.csv",
    }

PATHS = get_paths()
PATHS

{'db': PosixPath('/Users/rr/Library/Mobile Documents/com~apple~CloudDocs/05.Salvesen/00_DB/2026.duckdb'),
 'excel': PosixPath('/Users/rr/Library/Mobile Documents/com~apple~CloudDocs/05.Salvesen/Danone_Custos_v3.xlsx'),
 'parquet': PosixPath('/Users/rr/Library/Mobile Documents/com~apple~CloudDocs/05.Salvesen/inform_27_2026_final.parquet'),
 'viagens': PosixPath('/Users/rr/Library/Mobile Documents/com~apple~CloudDocs/05.Salvesen/viagens_2026.parquet'),
 'validacao': PosixPath('/Users/rr/Library/Mobile Documents/com~apple~CloudDocs/05.Salvesen/validacao_financeira_2026.csv'),
 'danone_diag': PosixPath('/Users/rr/Library/Mobile Documents/com~apple~CloudDocs/05.Salvesen/validacao_danone_2026.csv')}

## 01. Funções comuns

In [2]:
# ==========================
# 1. Datas
# ==========================
def normalizar_data(serie):
    texto = (
        serie.astype("string")
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
    )

    data = pd.to_datetime(
        texto,
        format="%Y%m%d",
        errors="coerce",
    )

    mask = data.isna()

    if mask.any():
        data.loc[mask] = pd.to_datetime(
            texto.loc[mask],
            errors="coerce",
        )

    return data

# ==========================
# 2. Referências
# ==========================
def normalizar_referencia(serie):
    return (
        serie.astype("string")
        .str.strip()
        .str.extract(r"(\d{10})", expand=False)
        .astype("string")
    )

# ==========================
# 3. Códigos postais
# ==========================
def normalizar_cp(serie):
    cp = (
        serie.astype("string")
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
        .str.replace(r"\D", "", regex=True)
        .str[:7]
    )

    invalido = (
        cp.str.len().fillna(0).lt(4)
        | cp.eq("0").fillna(False)
        | cp.eq("").fillna(False)
    )

    return cp.mask(invalido, pd.NA)

# ==========================
# 4. Divisão segura
# ==========================
def dividir_seguro(numerador, denominador):
    n = pd.to_numeric(numerador, errors="coerce")
    d = pd.to_numeric(denominador, errors="coerce")
    return n.div(d.where(d.ne(0)))

# ==========================
# 5. Moda segura
# ==========================
def moda_segura(serie):
    serie = serie.dropna()

    if serie.empty:
        return np.nan

    moda = serie.mode()

    if not moda.empty:
        return moda.iloc[0]

    return serie.iloc[0]

## 02. Regras de combustível

In [3]:
# ==========================
# 1. Taxas por CODACT e mês
# ==========================
taxas_ingresso = {
    11: {
        3: 0.0637,
        4: 0.1043,
        5: 0.0869,
        6: 0.0569,
        7: 0.0665,
        8: 0.1000,
    },

    74: {
        3: 0.0646,
        4: 0.1003,
        5: 0.0850,
        6: 0.0585,
        7: 0.0670,
        8: 0.0965,
    },

    84: {
        3: 0.0646,
        4: 0.1003,
        5: 0.0850,
        6: 0.0585,
        7: 0.0670,
        8: 0.0965,
    },

    118: {
        1: 0.0849,
        2: 0.0974,
        3: 0.1860,
        4: 0.2434,
        5: 0.2188,
        6: 0.1763,
        7: 0.1900,
        8: 0.2373,
    },

    131: {
        3: 0.0753,
        4: 0.1170,
        5: 0.0991,
        6: 0.0682,
        7: 0.0782,
        8: 0.1126,
    },

    262: {
        3: 0.0646,
        4: 0.1003,
        5: 0.0850,
        6: 0.0585,
        7: 0.0670,
        8: 0.0965,
    },

    275: {
        3: 0.0669,
        4: 0.1077,
        5: 0.0902,
        6: 0.0600,
        7: 0.0697,
        8: 0.1034,
    },

    280: {
        3: 0.0753,
        4: 0.1170,
        5: 0.0991,
        6: 0.0682,
        7: 0.0782,
        8: 0.1126,
    },

    282: {
        3: 0.0646,
        4: 0.1003,
        5: 0.0850,
        6: 0.0585,
        7: 0.0670,
        8: 0.0965,
    },

    332: {
        3: 0.0753,
        4: 0.1170,
        5: 0.0991,
        6: 0.0682,
        7: 0.0782,
        8: 0.1126,
    },

    391: {
        3: 0.0753,
        4: 0.1170,
        5: 0.0991,
        6: 0.0682,
        7: 0.0782,
        8: 0.1126,
    },

    398: {
        3: 0.0753,
        4: 0.1170,
        5: 0.0991,
        6: 0.0682,
        7: 0.0782,
        8: 0.1126,
    },

    403: {
        3: 0.0753,
        4: 0.1170,
        5: 0.0991,
        6: 0.0682,
        7: 0.0782,
        8: 0.1126,
    },

    404: {
        3: 0.0753,
        4: 0.1170,
        5: 0.0991,
        6: 0.0682,
        7: 0.0782,
        8: 0.1126,
    },

    405: {
        3: 0.0753,
        4: 0.1170,
        5: 0.0991,
        6: 0.0682,
        7: 0.0782,
        8: 0.1126,
    },

    415: {
        3: 0.0753,
        4: 0.1170,
        5: 0.0991,
        6: 0.0682,
        7: 0.0782,
        8: 0.1126,
    },

    419: {
        1: 0.0442,
        2: 0.0531,
        3: 0.1160,
        4: 0.1568,
        5: 0.1393,
        6: 0.1091,
        7: 0.1188,
        8: 0.1524,
    },

    429: {
        3: 0.0753,
        4: 0.1170,
        5: 0.0991,
        6: 0.0682,
        7: 0.0782,
        8: 0.1126,
    },

    588: {
        3: 0.0753,
        4: 0.1170,
        5: 0.0991,
        6: 0.0682,
        7: 0.0782,
        8: 0.1126,
    },

    660: {
        3: 0.0480,
        4: 0.0821,
        5: 0.0675,
        6: 0.0422,
        7: 0.0504,
        8: 0.0785,
    },

    680: {
        3: 0.0646,
        4: 0.1003,
        5: 0.0850,
        6: 0.0585,
        7: 0.0670,
        8: 0.0965,
    },

    784: {
        3: 0.0753,
        4: 0.1170,
        5: 0.0991,
        6: 0.0682,
        7: 0.0782,
        8: 0.1126,
    },
}

## 03. Importações

In [4]:
# ==========================
# 1. Validar fontes
# ==========================
if not PATHS["db"].exists():
    raise FileNotFoundError(
        f"DuckDB não encontrada: {PATHS['db']}"
    )

if not PATHS["excel"].exists():
    raise FileNotFoundError(
        f"Excel Danone não encontrado: {PATHS['excel']}"
    )

# ==========================
# 2. Ler DuckDB
# ==========================
with duckdb.connect(
    str(PATHS["db"]),
    read_only=True,
) as con:
    df_raw = con.sql("""
        SELECT *
        FROM inform_27_2026
    """).df()

    df_tarifas_raw = con.sql("""
        SELECT
            tipo_local,
            modelo_ingresso,
            tarifa_base,
            data_inicio,
            data_fim
        FROM Danone_Tarifas_2026
    """).df()

    coords_raw = con.sql("""
        SELECT
            cp AS CP,
            point_x AS POINT_X,
            point_y AS POINT_Y
        FROM coordenadas
    """).df()

# ==========================
# 3. Ler KILOSPO
# ==========================
raw_kilospo = pd.read_excel(
    PATHS["excel"],
    sheet_name="KILOSPO",
    header=None,
)

df_kilospo_raw = raw_kilospo.iloc[1:].copy()
df_kilospo_raw.columns = raw_kilospo.iloc[0]

# ==========================
# 4. Validar importações
# ==========================
pd.DataFrame({
    "fonte": [
        "inform_27_2026",
        "KILOSPO",
        "Danone_Tarifas_2026",
        "coordenadas",
    ],
    "linhas": [
        len(df_raw),
        len(df_kilospo_raw),
        len(df_tarifas_raw),
        len(coords_raw),
    ],
})

,fonte,linhas
0,inform_27_2026,779393
1,KILOSPO,44095
2,Danone_Tarifas_2026,38
3,coordenadas,178409


## 04. Universo financeiro — `inform_27_2026`

In [5]:
# ==========================
# 1. Cópia de trabalho
# ==========================
df_base = df_raw.copy()

# ==========================
# 2. Texto
# ==========================
for coluna in df_base.select_dtypes(
    include=["object"]
).columns:
    df_base[coluna] = (
        df_base[coluna]
        .astype("string")
        .str.strip()
    )

# ==========================
# 3. Numéricos
# ==========================
colunas_numericas = [
    "CODACT",
    "INGRESODT",
    "COSTEDT",
    "RENTADT",
    "PALETSDT",
    "PESO_BRUTO",
    "PALETS",
    "KM",
    "KMREALES",
    "CAMION_CAPACIDAD",
]

for coluna in colunas_numericas:
    if coluna in df_base.columns:
        df_base[coluna] = pd.to_numeric(
            df_base[coluna],
            errors="coerce",
        )

# ==========================
# 4. CODEUT
# ==========================
df_base["CODEUT"] = (
    df_base["CODEUT"]
    .astype("string")
    .str.strip()
    .replace({
        "": pd.NA,
        "nan": pd.NA,
        "None": pd.NA,
        "<NA>": pd.NA,
    })
)

# ==========================
# 5. Datas
# ==========================
df_base["FCARGA"] = normalizar_data(
    df_base["FCARGA"]
)

df_base["FENTREGA"] = normalizar_data(
    df_base["FENTREGA"]
)

# ==========================
# 6. Chave de referência
# ==========================
df_base["referencia_chave"] = (
    normalizar_referencia(
        df_base["REFERENCIA"]
    )
)

# ==========================
# 7. Guardar universo antes dos filtros
# ==========================
df_raw_check = df_base.copy()

# ==========================
# 8. Ajustes de ingresso
# ==========================
df_base["INGRESODT_ORIGINAL"] = (
    df_base["INGRESODT"]
)

df_base.loc[
    df_base["CODACT"].eq(11),
    "INGRESODT",
] = 0.0

mask_espanha = df_base["CODACT"].isin(
    CLIENTES_ESPANHA
)

df_base.loc[
    mask_espanha,
    "INGRESODT",
] = (
    df_base.loc[
        mask_espanha,
        "COSTEDT",
    ]
    * 1.05
)

# ==========================
# 9. Excluir CODACT 13
# ==========================
df_base = df_base.loc[
    ~df_base["CODACT"].isin(
        CODACT_EXCLUIR
    )
].copy()

# ==========================
# 10. Filtro operacional
# ==========================
mask_gestao = (
    df_base["GESTION"].eq("LIS")
    | (
        df_base["GESTION"].isna()
        & df_base["PROPIETARIO"].isin(
            ["CEP", "PTG"]
        )
    )
)

df_base = df_base.loc[
    mask_gestao
].copy()

# ==========================
# 11. ID e calendário
# ==========================
df_base["_linha_id"] = np.arange(
    len(df_base),
    dtype="int64",
)

df_base["data"] = df_base["FENTREGA"]
df_base["ano"] = df_base["data"].dt.year
df_base["mes"] = df_base["data"].dt.month
df_base["mes_nome"] = df_base["data"].dt.month_name()
df_base["week_number"] = (
    df_base["data"]
    .dt.isocalendar()
    .week
    .astype("Int64")
)
df_base["week_day"] = (
    df_base["data"]
    .dt.day_name()
)
df_base["mes_estrutura"] = (
    df_base["data"]
    .dt.to_period("M")
    .dt.to_timestamp()
)

# ==========================
# 12. Validação
# ==========================
pd.DataFrame({
    "metrica": [
        "Linhas raw",
        "Linhas universo financeiro",
        "CODACT 13 restantes",
        "CODEUT nulos",
        "FENTREGA nula",
    ],
    "valor": [
        len(df_raw),
        len(df_base),
        df_base["CODACT"].eq(13).sum(),
        df_base["CODEUT"].isna().sum(),
        df_base["FENTREGA"].isna().sum(),
    ],
})

,metrica,valor
0,Linhas raw,779393
1,Linhas universo financeiro,102291
2,CODACT 13 restantes,0
3,CODEUT nulos,0
4,FENTREGA nula,0


## 05. Danone — cálculo operacional

A KILOSPO é calculada integralmente.  
O ingresso financeiro só é reconhecido quando existe associação ao universo financeiro.

A diferença entre KILOSPO e `inform_27` é classificada, não forçada.

In [6]:
# ==========================
# 1. Preparar KILOSPO
# ==========================
df_danone = df_kilospo_raw.copy()

df_danone["referencia_chave"] = (
    normalizar_referencia(
        df_danone["PREFPE"]
    )
)

df_danone["PFEENT"] = normalizar_data(
    df_danone["PFEENT"]
)

df_danone["PCODCL"] = (
    df_danone["PCODCL"]
    .astype("string")
    .str.strip()
)

df_danone["tipo_local"] = (
    df_danone["Ruta Tte. Nueva"]
    .astype("string")
    .str.strip()
)

df_danone["Kg Neto Entregado"] = (
    pd.to_numeric(
        df_danone["Kg Neto Entregado"],
        errors="coerce",
    )
)

df_danone = df_danone.loc[
    df_danone["PFEENT"].dt.year.eq(
        ANO
    )
].copy()

# ==========================
# 2. Preparar tarifas
# ==========================
df_tarifas = df_tarifas_raw.copy()

df_tarifas["tipo_local"] = (
    df_tarifas["tipo_local"]
    .astype("string")
    .str.strip()
)

df_tarifas["modelo_ingresso"] = (
    df_tarifas["modelo_ingresso"]
    .astype("string")
    .str.strip()
)

df_tarifas["tarifa_base"] = (
    pd.to_numeric(
        df_tarifas["tarifa_base"],
        errors="coerce",
    )
)

df_tarifas["data_inicio"] = (
    pd.to_datetime(
        df_tarifas["data_inicio"],
        errors="coerce",
    )
)

df_tarifas["data_fim"] = (
    pd.to_datetime(
        df_tarifas["data_fim"],
        errors="coerce",
    )
)

df_tarifas_ativas = (
    df_tarifas.loc[
        df_tarifas["data_fim"].isna()
    ]
    .copy()
)

if (
    df_tarifas_ativas["tipo_local"]
    .duplicated()
    .any()
):
    raise ValueError(
        "Existem tarifas activas duplicadas por tipo_local."
    )

# ==========================
# 3. Associar tarifa
# ==========================
df_danone = df_danone.merge(
    df_tarifas_ativas[
        [
            "tipo_local",
            "modelo_ingresso",
            "tarifa_base",
        ]
    ],
    on="tipo_local",
    how="left",
    validate="many_to_one",
)

# ==========================
# 4. Ingresso base
# ==========================
df_danone["ingresso_danone_base"] = (
    pd.Series(
        pd.NA,
        index=df_danone.index,
        dtype="Float64",
    )
)

mask_ton = (
    df_danone["modelo_ingresso"]
    .eq("POR_TONELADA")
    .fillna(False)
)

mask_entrega = (
    df_danone["modelo_ingresso"]
    .eq("POR_ENTREGA")
    .fillna(False)
)

mask_sem = (
    df_danone["modelo_ingresso"]
    .eq("SEM_INGRESSO")
    .fillna(False)
)

df_danone.loc[
    mask_ton,
    "ingresso_danone_base",
] = (
    df_danone.loc[
        mask_ton,
        "Kg Neto Entregado",
    ]
    / 1000
    * df_danone.loc[
        mask_ton,
        "tarifa_base",
    ]
)

df_danone.loc[
    mask_entrega,
    "ingresso_danone_base",
] = (
    df_danone.loc[
        mask_entrega,
        "tarifa_base",
    ]
)

df_danone.loc[
    mask_sem,
    "ingresso_danone_base",
] = 0.0

# ==========================
# 5. Extra mensal
# ==========================
df_danone["mes_periodo"] = (
    df_danone["PFEENT"]
    .dt.to_period("M")
)

df_danone["extra_danone"] = 0.0

mask_extra = (
    df_danone["ingresso_danone_base"]
    .notna()
    & df_danone["ingresso_danone_base"]
    .gt(0)
    & df_danone["mes_periodo"]
    .notna()
)

total_base_mes = (
    df_danone["ingresso_danone_base"]
    .where(mask_extra)
    .groupby(
        df_danone["mes_periodo"]
    )
    .transform("sum")
)

if EXTRA_DANONE_MENSAL != 0:
    df_danone.loc[
        mask_extra,
        "extra_danone",
    ] = (
        df_danone.loc[
            mask_extra,
            "ingresso_danone_base",
        ]
        .div(
            total_base_mes.loc[
                mask_extra
            ]
        )
        .mul(EXTRA_DANONE_MENSAL)
    )

df_danone["ingresso_danone"] = (
    df_danone["ingresso_danone_base"]
    + df_danone["extra_danone"]
)

# ==========================
# 6. Estado do cálculo
# ==========================
condicoes = [
    (
        df_danone["modelo_ingresso"]
        .eq("POR_TONELADA")
        & df_danone["ingresso_danone"]
        .notna()
    ).fillna(False).to_numpy(dtype=bool),

    (
        df_danone["modelo_ingresso"]
        .eq("POR_ENTREGA")
        & df_danone["ingresso_danone"]
        .notna()
    ).fillna(False).to_numpy(dtype=bool),

    df_danone["modelo_ingresso"]
    .eq("SEM_INGRESSO")
    .fillna(False)
    .to_numpy(dtype=bool),

    df_danone["modelo_ingresso"]
    .eq("CAPILAR")
    .fillna(False)
    .to_numpy(dtype=bool),

    df_danone["modelo_ingresso"]
    .eq("POR_DEFINIR")
    .fillna(False)
    .to_numpy(dtype=bool),

    df_danone["modelo_ingresso"]
    .isna()
    .to_numpy(dtype=bool),
]

escolhas = [
    "CALCULADO_POR_TONELADA",
    "CALCULADO_POR_ENTREGA",
    "SEM_INGRESSO",
    "CAPILAR",
    "POR_DEFINIR",
    "SEM_TARIFA",
]

df_danone["estado_calculo"] = (
    np.select(
        condicoes,
        escolhas,
        default="REVER",
    )
)

# ==========================
# 7. Validação mensal
# ==========================
validacao_danone_mes = (
    df_danone
    .groupby(
        "mes_periodo",
        as_index=False,
    )
    .agg(
        linhas=(
            "referencia_chave",
            "size",
        ),
        ingresso_base=(
            "ingresso_danone_base",
            lambda x: x.sum(
                min_count=1
            ),
        ),
        extra=(
            "extra_danone",
            "sum",
        ),
        ingresso_final=(
            "ingresso_danone",
            lambda x: x.sum(
                min_count=1
            ),
        ),
    )
)

validacao_danone_mes

,mes_periodo,linhas,ingresso_base,extra,ingresso_final
0,2026-01,5522,122884.131421,20000.0,142884.131421
1,2026-02,5387,112978.531191,20000.0,132978.531191
2,2026-03,5730,133113.817669,20000.0,153113.817669
3,2026-04,5309,135126.357042,20000.0,155126.357042
4,2026-05,5293,133374.048236,20000.0,153374.048236
5,2026-06,5405,145315.388264,20000.0,165315.388264
6,2026-07,6072,168843.085034,20000.0,188843.085034
7,2026-08,5159,147566.346052,20000.0,167566.346052
8,2026-09,218,3999.145015,20000.0,23999.145015


## 06. Associação Danone e reconciliação

In [7]:
# ==========================
# 1. Agregar por referência/data
# ==========================
danone_chave = (
    df_danone.loc[
        df_danone["referencia_chave"]
        .notna()
        & df_danone["PFEENT"]
        .notna()
    ]
    .groupby(
        [
            "referencia_chave",
            "PFEENT",
        ],
        as_index=False,
    )
    .agg(
        ingresso_danone_total=(
            "ingresso_danone",
            lambda x: x.sum(
                min_count=1
            ),
        ),
        ingresso_danone_base_total=(
            "ingresso_danone_base",
            lambda x: x.sum(
                min_count=1
            ),
        ),
        extra_danone_total=(
            "extra_danone",
            "sum",
        ),
    )
)

# ==========================
# 2. Associar exacto
# ==========================
df = df_base.copy()

df = df.merge(
    danone_chave,
    left_on=[
        "referencia_chave",
        "FENTREGA",
    ],
    right_on=[
        "referencia_chave",
        "PFEENT",
    ],
    how="left",
    validate="many_to_one",
)

# ==========================
# 3. Evitar duplicação
# ==========================
tem_match = (
    df["ingresso_danone_total"]
    .notna()
)

ultima_linha = (
    tem_match
    & ~df.duplicated(
        subset=[
            "referencia_chave",
            "FENTREGA",
        ],
        keep="last",
    )
)

df["ingresso_danone"] = (
    pd.Series(
        pd.NA,
        index=df.index,
        dtype="Float64",
    )
)

df.loc[
    ultima_linha,
    "ingresso_danone",
] = df.loc[
    ultima_linha,
    "ingresso_danone_total",
]

df = df.drop(
    columns=[
        "PFEENT",
        "ingresso_danone_total",
        "ingresso_danone_base_total",
        "extra_danone_total",
    ],
    errors="ignore",
)

# ==========================
# 4. Ingresso total
# ==========================
df["total_ingresso"] = (
    df["INGRESODT"].fillna(0.0)
    + df["ingresso_danone"].fillna(0.0)
)

In [8]:
# ==========================
# 1. Âmbito de validação
# ==========================
danone_validacao = (
    danone_chave.loc[
        danone_chave["PFEENT"]
        .dt.month
        .le(MES_VALIDACAO_DANONE_MAX)
    ]
    .copy()
)

base_validacao = (
    df_base.loc[
        df_base["FENTREGA"]
        .dt.month
        .le(MES_VALIDACAO_DANONE_MAX)
    ]
    .copy()
)

raw_validacao = (
    df_raw_check.loc[
        df_raw_check["FENTREGA"]
        .dt.month
        .le(MES_VALIDACAO_DANONE_MAX)
        | df_raw_check["FENTREGA"].isna()
    ]
    .copy()
)

# ==========================
# 2. Chaves disponíveis
# ==========================
chaves_base_exactas = set(
    zip(
        base_validacao["referencia_chave"],
        base_validacao["FENTREGA"],
    )
)

refs_base = set(
    base_validacao[
        "referencia_chave"
    ].dropna()
)

refs_raw = set(
    raw_validacao[
        "referencia_chave"
    ].dropna()
)

refs_codact13 = set(
    raw_validacao.loc[
        raw_validacao["CODACT"].eq(13),
        "referencia_chave",
    ].dropna()
)

# ==========================
# 3. Classificar diferença
# ==========================
def classificar_danone(row):
    chave = (
        row["referencia_chave"],
        row["PFEENT"],
    )

    ref = row["referencia_chave"]

    if chave in chaves_base_exactas:
        return "ASSOCIADO_EXACTO"

    if ref in refs_codact13:
        return "EXCLUIDO_CODACT_13"

    if ref in refs_base:
        return "REFERENCIA_EXISTE_DATA_DIFERENTE"

    if ref in refs_raw:
        return "EXCLUIDO_OUTRO_FILTRO"

    return "FORA_INFORM27"

danone_validacao[
    "estado_reconciliacao"
] = danone_validacao.apply(
    classificar_danone,
    axis=1,
)

# ==========================
# 4. Resumo
# ==========================
validacao_danone = (
    danone_validacao
    .groupby(
        "estado_reconciliacao",
        as_index=False,
    )
    .agg(
        casos=(
            "referencia_chave",
            "size",
        ),
        referencias=(
            "referencia_chave",
            "nunique",
        ),
        ingresso_base=(
            "ingresso_danone_base_total",
            "sum",
        ),
        extra=(
            "extra_danone_total",
            "sum",
        ),
        ingresso=(
            "ingresso_danone_total",
            "sum",
        ),
    )
    .sort_values(
        "ingresso",
        ascending=False,
    )
    .reset_index(drop=True)
)

# ==========================
# 5. Reconciliação
# ==========================
total_kilospo = (
    danone_validacao[
        "ingresso_danone_total"
    ].sum()
)

total_classificado = (
    validacao_danone[
        "ingresso"
    ].sum()
)

erro_reconciliacao_danone = (
    total_kilospo
    - total_classificado
)

validacao_danone

,estado_reconciliacao,casos,referencias,ingresso_base,extra,ingresso
0,ASSOCIADO_EXACTO,13866,13866,1050288.036634,152965.109168,1203253.145802
1,EXCLUIDO_CODACT_13,24565,24564,41308.584438,5962.101594,47270.686032
2,FORA_INFORM27,2165,2165,5491.475694,779.558457,6271.03415
3,REFERENCIA_EXISTE_DATA_DIFERENTE,83,83,2112.202415,293.022721,2405.225136


## 07. Combustível

In [9]:
# ==========================
# 1. Mês da taxa
# ==========================
df["mes_taxa_combustivel"] = (
    df["FENTREGA"].dt.month
    + DESFASAMENTO_MES_COMBUSTIVEL
)

# ==========================
# 2. Taxa
# ==========================
def obter_taxa(codact, mes):
    if pd.isna(codact) or pd.isna(mes):
        return np.nan

    mes = int(mes)

    if mes < 1 or mes > 12:
        return np.nan

    return taxas_ingresso.get(
        int(codact),
        {},
    ).get(
        mes,
        np.nan,
    )

df["taxa_combustivel"] = [
    obter_taxa(codact, mes)
    for codact, mes in zip(
        df["CODACT"],
        df["mes_taxa_combustivel"],
    )
]

df["taxa_combustivel"] = (
    pd.to_numeric(
        df["taxa_combustivel"],
        errors="coerce",
    )
)

# ==========================
# 3. Base combustível
# ==========================
df["base_combustivel"] = (
    pd.to_numeric(
        df["INGRESODT"],
        errors="coerce",
    )
)

mask_codact11 = (
    df["CODACT"].eq(11)
)

df.loc[
    mask_codact11,
    "base_combustivel",
] = (
    pd.to_numeric(
        df.loc[
            mask_codact11,
            "ingresso_danone",
        ],
        errors="coerce",
    )
)

# ==========================
# 4. Acerto combustível
# ==========================
df["acerto_combustivel"] = 0.0

mask_taxa = (
    df["taxa_combustivel"].notna()
    & df["base_combustivel"].notna()
)

df.loc[
    mask_taxa,
    "acerto_combustivel",
] = (
    df.loc[
        mask_taxa,
        "base_combustivel",
    ]
    * df.loc[
        mask_taxa,
        "taxa_combustivel",
    ]
)

df["total_ingresso_c"] = (
    df["total_ingresso"]
    + df["acerto_combustivel"]
)

## 08. Capacidade e coordenadas

In [10]:
# ==========================
# 1. Capacidade
# ==========================
df["CAMION_CAPACIDAD_NUM"] = (
    pd.to_numeric(
        df["CAMION_CAPACIDAD"],
        errors="coerce",
    )
)

df["capacidade_norm"] = (
    df["CAMION_CAPACIDAD_NUM"]
    .map(MAPA_CAPACIDADE)
)

capacidade_codeut = (
    df.loc[
        df["CODEUT"].notna()
        & df["capacidade_norm"].notna()
    ]
    .groupby("CODEUT")[
        "capacidade_norm"
    ]
    .agg(moda_segura)
)

mask_sem_capacidade = (
    df["CODEUT"].notna()
    & df["capacidade_norm"].isna()
)

df.loc[
    mask_sem_capacidade,
    "capacidade_norm",
] = (
    df.loc[
        mask_sem_capacidade,
        "CODEUT",
    ]
    .map(capacidade_codeut)
)

df["dados_validos"] = (
    df["CODEUT"].notna()
    & df["FENTREGA"].notna()
    & df["capacidade_norm"].notna()
)

In [11]:
# ==========================
# 1. Preparar coordenadas
# ==========================
coords = coords_raw.copy()

coords["POINT_X"] = pd.to_numeric(
    coords["POINT_X"]
    .astype("string")
    .str.strip()
    .str.replace(",", ".", regex=False),
    errors="coerce",
)

coords["POINT_Y"] = pd.to_numeric(
    coords["POINT_Y"]
    .astype("string")
    .str.strip()
    .str.replace(",", ".", regex=False),
    errors="coerce",
)

coords["CP_chave"] = normalizar_cp(
    coords["CP"]
)

coords["CP_parte"] = (
    coords["CP_chave"]
    .str[:4]
)

df["CPOSTAL_chave"] = normalizar_cp(
    df["CPOSTAL"]
)

df["CPOSTAD_chave"] = normalizar_cp(
    df["CPOSTAD"]
)

df["CPOSTAL_parte"] = (
    df["CPOSTAL_chave"]
    .str[:4]
)

df["CPOSTAD_parte"] = (
    df["CPOSTAD_chave"]
    .str[:4]
)

# ==========================
# 2. CP exacto
# ==========================
coords_cp = (
    coords.dropna(
        subset=[
            "CP_chave",
            "POINT_X",
            "POINT_Y",
        ]
    )
    .groupby(
        "CP_chave",
        as_index=False,
    )[
        ["POINT_X", "POINT_Y"]
    ]
    .mean()
)

origem = coords_cp.rename(
    columns={
        "CP_chave": "CPOSTAL_chave",
        "POINT_X": "longitude_origem",
        "POINT_Y": "latitude_origem",
    }
)

destino = coords_cp.rename(
    columns={
        "CP_chave": "CPOSTAD_chave",
        "POINT_X": "longitude_destino",
        "POINT_Y": "latitude_destino",
    }
)

df = df.merge(
    origem,
    on="CPOSTAL_chave",
    how="left",
    validate="many_to_one",
)

df = df.merge(
    destino,
    on="CPOSTAD_chave",
    how="left",
    validate="many_to_one",
)

# ==========================
# 3. Fallback 4 dígitos
# ==========================
centroid = (
    coords.dropna(
        subset=[
            "CP_parte",
            "POINT_X",
            "POINT_Y",
        ]
    )
    .groupby(
        "CP_parte",
        as_index=False,
    )[
        ["POINT_X", "POINT_Y"]
    ]
    .mean()
)

centroid_origem = centroid.rename(
    columns={
        "CP_parte": "CPOSTAL_parte",
        "POINT_X": "longitude_origem_fb",
        "POINT_Y": "latitude_origem_fb",
    }
)

centroid_destino = centroid.rename(
    columns={
        "CP_parte": "CPOSTAD_parte",
        "POINT_X": "longitude_destino_fb",
        "POINT_Y": "latitude_destino_fb",
    }
)

df = df.merge(
    centroid_origem,
    on="CPOSTAL_parte",
    how="left",
    validate="many_to_one",
)

df = df.merge(
    centroid_destino,
    on="CPOSTAD_parte",
    how="left",
    validate="many_to_one",
)

df["longitude_origem"] = (
    df["longitude_origem"]
    .fillna(
        df["longitude_origem_fb"]
    )
)

df["latitude_origem"] = (
    df["latitude_origem"]
    .fillna(
        df["latitude_origem_fb"]
    )
)

df["longitude_destino"] = (
    df["longitude_destino"]
    .fillna(
        df["longitude_destino_fb"]
    )
)

df["latitude_destino"] = (
    df["latitude_destino"]
    .fillna(
        df["latitude_destino_fb"]
    )
)

df = df.drop(
    columns=[
        "longitude_origem_fb",
        "latitude_origem_fb",
        "longitude_destino_fb",
        "latitude_destino_fb",
    ],
    errors="ignore",
)

## 09. Custos, viagens e rentabilidade

In [12]:
# ==========================
# 1. Preparar valores
# ==========================
df["PALETS"] = pd.to_numeric(
    df["PALETS"],
    errors="coerce",
).fillna(0.0)

df["COSTEDT"] = pd.to_numeric(
    df["COSTEDT"],
    errors="coerce",
).fillna(0.0)

# ==========================
# 2. Paletes viagem/mês
# ==========================
paletes_codeut_mes = (
    df.groupby(
        [
            "mes_estrutura",
            "CODEUT",
        ],
        dropna=False,
    )["PALETS"]
    .transform("sum")
)

mask_estrutura = (
    df["mes_estrutura"].notna()
    & df["CODEUT"].notna()
    & paletes_codeut_mes.gt(0)
)

# ==========================
# 3. Viagens mensais
# ==========================
n_codeut_mes = (
    df.loc[
        mask_estrutura
    ]
    .groupby(
        "mes_estrutura"
    )["CODEUT"]
    .nunique()
)

df["n_viagens_estrutura_mes"] = (
    df["mes_estrutura"]
    .map(n_codeut_mes)
)

df["custo_estrutura_viagem"] = (
    CUSTO_ESTRUTURA_MENSAL
    / df["n_viagens_estrutura_mes"]
)

# ==========================
# 4. Rateio estrutura
# ==========================
df["custo_estrutura"] = 0.0

df.loc[
    mask_estrutura,
    "custo_estrutura",
] = (
    df.loc[
        mask_estrutura,
        "custo_estrutura_viagem",
    ]
    * df.loc[
        mask_estrutura,
        "PALETS",
    ]
    / paletes_codeut_mes.loc[
        mask_estrutura
    ]
)

# ==========================
# 5. Custo e margem
# ==========================
df["custo_total_c"] = (
    df["COSTEDT"]
    + df["custo_estrutura"]
)

df["margem"] = (
    df["total_ingresso_c"]
    - df["custo_total_c"]
)

df["custo_por_palete"] = (
    dividir_seguro(
        df["custo_total_c"],
        df["PALETS"],
    )
)

df["ingresso_por_palete"] = (
    dividir_seguro(
        df["total_ingresso_c"],
        df["PALETS"],
    )
)

df["margem_por_palete"] = (
    dividir_seguro(
        df["margem"],
        df["PALETS"],
    )
)

df["margem_pct"] = (
    dividir_seguro(
        df["margem"],
        df["total_ingresso_c"],
    )
    * 100
)

In [13]:
# ==========================
# 1. Tabela oficial de viagens
# ==========================
df_viagens = (
    df.loc[
        df["CODEUT"].notna()
    ]
    .groupby(
        "CODEUT",
        as_index=False,
    )
    .agg(
        data=("FENTREGA", "min"),
        mes_estrutura=(
            "mes_estrutura",
            "min",
        ),
        TRANSPORTISTA=(
            "TRANSPORTISTA",
            "first",
        ),
        LOCORIGEN=(
            "LOCORIGEN",
            "first",
        ),
        LOCDESTINO=(
            "LOCDESTINO",
            "first",
        ),
        paletes=("PALETS", "sum"),
        peso_kg=("PESO_BRUTO", "sum"),
        custo_transporte=(
            "COSTEDT",
            "sum",
        ),
        custo_estrutura=(
            "custo_estrutura",
            "sum",
        ),
        custo_total=(
            "custo_total_c",
            "sum",
        ),
        ingresso_total=(
            "total_ingresso_c",
            "sum",
        ),
        capacidade_original=(
            "capacidade_norm",
            moda_segura,
        ),
    )
)

# ==========================
# 2. Capacidade real
# ==========================
df_viagens["capacidade_real"] = (
    df_viagens["capacidade_original"]
)

mask_duplo_deck = (
    df_viagens["capacidade_original"]
    .eq(33)
    .fillna(False)
    & df_viagens["paletes"].gt(40)
    & df_viagens["paletes"].le(55)
)

df_viagens.loc[
    mask_duplo_deck,
    "capacidade_real",
] = 55

# ==========================
# 3. Estado ocupação
# ==========================
condicoes_ocupacao = [
    df_viagens["capacidade_real"]
    .isna()
    .to_numpy(dtype=bool),

    df_viagens["capacidade_real"]
    .le(0)
    .fillna(False)
    .to_numpy(dtype=bool),

    df_viagens["paletes"]
    .gt(
        df_viagens["capacidade_real"]
    )
    .fillna(False)
    .to_numpy(dtype=bool),
]

df_viagens["ocupacao_estado"] = (
    np.select(
        condicoes_ocupacao,
        [
            "SEM_CAPACIDADE",
            "CAPACIDADE_INVALIDA",
            "EXCESSO_CAPACIDADE",
        ],
        default="OK",
    )
)

# ==========================
# 4. Ocupação
# ==========================
df_viagens["ocupacao_pct"] = (
    dividir_seguro(
        df_viagens["paletes"],
        df_viagens["capacidade_real"],
    )
    * 100
)

df_viagens.loc[
    ~df_viagens[
        "ocupacao_estado"
    ].eq("OK"),
    "ocupacao_pct",
] = np.nan

# ==========================
# 5. KPIs viagem
# ==========================
df_viagens["peso_ton"] = (
    df_viagens["peso_kg"]
    / 1000
)

df_viagens["custo_por_palete"] = (
    dividir_seguro(
        df_viagens["custo_total"],
        df_viagens["paletes"],
    )
)

df_viagens[
    "ingresso_por_palete"
] = dividir_seguro(
    df_viagens["ingresso_total"],
    df_viagens["paletes"],
)

df_viagens["custo_por_ton"] = (
    dividir_seguro(
        df_viagens["custo_total"],
        df_viagens["peso_ton"],
    )
)

df_viagens[
    "ingresso_por_ton"
] = dividir_seguro(
    df_viagens["ingresso_total"],
    df_viagens["peso_ton"],
)

df_viagens["margem"] = (
    df_viagens["ingresso_total"]
    - df_viagens["custo_total"]
)

df_viagens["margem_pct"] = (
    dividir_seguro(
        df_viagens["margem"],
        df_viagens["ingresso_total"],
    )
    * 100
)

# ==========================
# 6. Associar ao detalhe
# ==========================
df = df.merge(
    df_viagens[
        [
            "CODEUT",
            "capacidade_real",
            "ocupacao_pct",
            "ocupacao_estado",
        ]
    ],
    on="CODEUT",
    how="left",
    validate="many_to_one",
)

## 10. Validações finais

In [14]:
# ==========================
# 1. Estrutura mensal
# ==========================
validacao_estrutura = (
    df.loc[
        df["mes_estrutura"].isin(
            n_codeut_mes.index
        )
    ]
    .groupby(
        "mes_estrutura",
        as_index=False,
    )
    .agg(
        viagens=(
            "CODEUT",
            "nunique",
        ),
        estrutura=(
            "custo_estrutura",
            "sum",
        ),
    )
)

validacao_estrutura["esperado"] = (
    CUSTO_ESTRUTURA_MENSAL
)

validacao_estrutura["erro"] = (
    validacao_estrutura["estrutura"]
    - validacao_estrutura["esperado"]
)

erro_estrutura = (
    validacao_estrutura["erro"]
    .abs()
    .max()
)

if pd.isna(erro_estrutura):
    erro_estrutura = 0.0

# ==========================
# 2. Consistência detalhe/viagens
# ==========================
erro_custo = abs(
    df["custo_total_c"].sum()
    - df_viagens["custo_total"].sum()
)

erro_ingresso = abs(
    df["total_ingresso_c"].sum()
    - df_viagens["ingresso_total"].sum()
)

erro_margem = (
    (
        df["margem"]
        - (
            df["total_ingresso_c"]
            - df["custo_total_c"]
        )
    )
    .abs()
    .max()
)

if pd.isna(erro_margem):
    erro_margem = 0.0

# ==========================
# 3. Controlos finais
# ==========================
validacao_financeira = pd.DataFrame({
    "controlo": [
        "Reconciliação Danone classificada",
        "Estrutura mensal",
        "Custo detalhe = viagens",
        "Ingresso detalhe = viagens",
        "CODEUT texto nan",
        "CODEUT único em viagens",
        "Ocupação OK > 100%",
        "Margem consistente",
    ],
    "valor": [
        abs(erro_reconciliacao_danone),
        float(erro_estrutura),
        float(erro_custo),
        float(erro_ingresso),
        int(df["CODEUT"].eq("nan").sum()),
        int(
            df_viagens["CODEUT"]
            .duplicated()
            .sum()
        ),
        int(
            df_viagens.loc[
                df_viagens[
                    "ocupacao_estado"
                ].eq("OK"),
                "ocupacao_pct",
            ]
            .gt(100 + 1e-9)
            .sum()
        ),
        float(erro_margem),
    ],
    "tolerancia": [
        TOLERANCIA_FINANCEIRA,
        TOLERANCIA_FINANCEIRA,
        TOLERANCIA_FINANCEIRA,
        TOLERANCIA_FINANCEIRA,
        0,
        0,
        0,
        TOLERANCIA_FINANCEIRA,
    ],
})

validacao_financeira["estado"] = (
    np.where(
        validacao_financeira["valor"]
        <= validacao_financeira[
            "tolerancia"
        ],
        "OK",
        "ERRO",
    )
)

validacao_financeira

,controlo,valor,tolerancia,estado
0,Reconciliação Danone classificada,5.820766e-09,0.01,OK
1,Estrutura mensal,1.455192e-11,0.01,OK
2,Custo detalhe = viagens,0.000000e+00,0.01,OK
3,Ingresso detalhe = viagens,9.313226e-10,0.01,OK
4,CODEUT texto nan,0.000000e+00,0.00,OK
5,CODEUT único em viagens,0.000000e+00,0.00,OK
6,Ocupação OK > 100%,0.000000e+00,0.00,OK
7,Margem consistente,0.000000e+00,0.01,OK


In [15]:
# ==========================
# 1. Validar bloqueios
# ==========================
erros_criticos = (
    validacao_financeira.loc[
        validacao_financeira[
            "estado"
        ].eq("ERRO")
    ]
    .copy()
)

if not erros_criticos.empty:
    raise ValueError(
        "Existem validações finais com ERRO. "
        "Corrige-as antes da exportação."
    )

## 11. Dataset final e exportação

In [16]:
# ==========================
# 1. Colunas finais
# ==========================
colunas_finais = [
    "REFERENCIA",
    "referencia_chave",
    "FENTREGA",
    "FCARGA",
    "CODEUT",
    "capacidade_norm",
    "capacidade_real",
    "ocupacao_pct",
    "ocupacao_estado",
    "PALETS",
    "PESO_BRUTO",
    "INGRESODT",
    "ingresso_danone",
    "total_ingresso",
    "taxa_combustivel",
    "base_combustivel",
    "acerto_combustivel",
    "total_ingresso_c",
    "COSTEDT",
    "custo_estrutura",
    "custo_total_c",
    "margem",
    "margem_pct",
    "custo_por_palete",
    "ingresso_por_palete",
    "margem_por_palete",
    "PROV_ORIGEN",
    "LOCORIGEN",
    "PROV_DESTINO",
    "LOCDESTINO",
    "CPOSTAL",
    "CPOSTAD",
    "longitude_origem",
    "latitude_origem",
    "longitude_destino",
    "latitude_destino",
    "TRANSPORTISTA",
    "week_day",
    "week_number",
    "ano",
    "mes",
    "mes_nome",
    "data",
    "mes_estrutura",
    "dados_validos",
    "LOCCAR",
    "LUGARCARGA",
    "LOCDES",
    "LUGARDESCARGA",
    "TRACTORA",
    "CODEDT",
    "CODACT",
    "TIPOCLIENTE",
    "TIPOFLUJO",
    "PROV_ENTREGAR",
    "PAISENTREGAR",
    "ACTIVIDAD",
]

colunas_existentes = [
    coluna
    for coluna in colunas_finais
    if coluna in df.columns
]

df_final = (
    df[colunas_existentes]
    .copy()
)

df_final.head()

,REFERENCIA,referencia_chave,FENTREGA,FCARGA,CODEUT,capacidade_norm,capacidade_real,ocupacao_pct,ocupacao_estado,PALETS,...,LOCDES,LUGARDESCARGA,TRACTORA,CODEDT,CODACT,TIPOCLIENTE,TIPOFLUJO,PROV_ENTREGAR,PAISENTREGAR,ACTIVIDAD
0,5021950798 414190635,5021950798,2026-02-02,2026-02-01,3365589,33.0,33.0,75.757576,OK,1.0,...,319558,3. MALAQUIAS - CASH & CARRY O. AZ,93-QI-75,25531441,11,TLD PORTUGAL,Directo,Aveiro,PORTUGAL,DANONE PORTUGAL
1,2021792046,2021792046,2026-02-02,2026-02-01,3365589,33.0,33.0,75.757576,OK,1.0,...,298515,DL Transportes Fernando Simões Monteiro Unip. Lda,93-QI-75,25532252,757,<NA>,Directo,Coimbra,PORTUGAL,SUMOLCOMPAL MARKETING
2,5021982094 414203894,5021982094,2026-02-02,2026-02-01,3365589,33.0,33.0,75.757576,OK,2.0,...,319783,"3. MARABUTO-PRODUT.ALIMENTARES,SA",93-QI-75,25532753,11,TLD PORTUGAL,Directo,Aveiro,PORTUGAL,DANONE PORTUGAL
3,5021963392 414182550,5021963392,2026-02-02,2026-02-01,3365589,33.0,33.0,75.757576,OK,1.0,...,319201,3. COOPERATIVA AGRICOLA DA TOCHA,93-QI-75,25534995,11,TLD PORTUGAL,Directo,Coimbra,PORTUGAL,DANONE PORTUGAL
4,5021973560 414166769,5021973560,2026-02-02,2026-01-30,3365590,33.0,33.0,<NA>,EXCESSO_CAPACIDADE,1.0,...,328547,DL TRANSPARENTODISSEIA - Transportes Unipessoa...,0000XXX,25515326,11,TLD PORTUGAL,Directo,Faro,PORTUGAL,DANONE PORTUGAL


In [17]:
# ==========================
# 1. Exportar detalhe
# ==========================
df_final.to_parquet(
    PATHS["parquet"],
    index=False,
)

# ==========================
# 2. Exportar viagens
# ==========================
df_viagens.to_parquet(
    PATHS["viagens"],
    index=False,
)

# ==========================
# 3. Exportar validações
# ==========================
validacao_financeira.to_csv(
    PATHS["validacao"],
    index=False,
    sep=";",
    decimal=",",
)

validacao_danone.to_csv(
    PATHS["danone_diag"],
    index=False,
    sep=";",
    decimal=",",
)

# ==========================
# 4. Resultado
# ==========================
pd.DataFrame({
    "output": [
        "Detalhe Power BI",
        "Viagens",
        "Validação financeira",
        "Reconciliação Danone",
    ],
    "ficheiro": [
        str(PATHS["parquet"]),
        str(PATHS["viagens"]),
        str(PATHS["validacao"]),
        str(PATHS["danone_diag"]),
    ],
    "linhas": [
        len(df_final),
        len(df_viagens),
        len(validacao_financeira),
        len(validacao_danone),
    ],
})

,output,ficheiro,linhas
0,Detalhe Power BI,/Users/rr/Library/Mobile Documents/com~apple~C...,102291
1,Viagens,/Users/rr/Library/Mobile Documents/com~apple~C...,16156
2,Validação financeira,/Users/rr/Library/Mobile Documents/com~apple~C...,8
3,Reconciliação Danone,/Users/rr/Library/Mobile Documents/com~apple~C...,4


# 12. Análises

A partir deste ponto não são alteradas regras de negócio nem valores do pipeline.  
As análises usam apenas `df_final`, `df_viagens` e as tabelas de validação já produzidas.

In [18]:
# ==========================
# 1. Resumo financeiro mensal
# ==========================
resumo_mensal = (
    df_final
    .groupby(
        "mes_estrutura",
        as_index=False,
    )
    .agg(
        linhas=("CODEUT", "size"),
        viagens=("CODEUT", "nunique"),
        paletes=("PALETS", "sum"),
        peso_kg=("PESO_BRUTO", "sum"),
        ingresso=(
            "total_ingresso_c",
            "sum",
        ),
        custo=(
            "custo_total_c",
            "sum",
        ),
        margem=("margem", "sum"),
        estrutura=(
            "custo_estrutura",
            "sum",
        ),
        combustivel=(
            "acerto_combustivel",
            "sum",
        ),
    )
)

resumo_mensal["peso_ton"] = (
    resumo_mensal["peso_kg"]
    / 1000
)

resumo_mensal["margem_pct"] = (
    dividir_seguro(
        resumo_mensal["margem"],
        resumo_mensal["ingresso"],
    )
    * 100
)

resumo_mensal

,mes_estrutura,linhas,viagens,paletes,peso_kg,ingresso,custo,margem,estrutura,combustivel,peso_ton,margem_pct
0,2026-01-01,11327,1923,46298.001,10473868.763,464356.71835,513737.91,-49381.19165,103000.0,0.000000,10473.868763,-10.634323
1,2026-02-01,10645,1718,38238.75,8952691.761,411041.180994,458867.4,-47826.219006,103000.0,2058.953390,8952.691761,-11.635384
2,2026-03-01,12281,1966,46487.563,12553858.055,508804.760301,540931.2,-32126.439699,103000.0,2926.743206,12553.858055,-6.3141
3,2026-04-01,12577,1972,48077.75,11656719.178,558217.411672,572365.89,-14148.478328,103000.0,30094.108710,11656.719178,-2.534582
4,2026-05-01,12581,1953,47682.05,11801330.019,564403.313304,573870.16,-9466.846696,103000.0,45637.587756,11801.330019,-1.677319
5,2026-06-01,12460,1966,45447.16,11952113.721,551492.711347,580716.09,-29223.378653,103000.0,40162.106093,11952.113721,-5.29896
6,2026-07-01,14179,2167,49278.5,13429174.515,651931.579208,643897.92,8033.659208,103000.0,33775.387012,13429.174515,1.232286
7,2026-08-01,13273,2019,49251.0,12551971.754,553292.0225,555267.22,-1975.1975,103000.0,32553.613206,12551.971754,-0.35699
8,2026-09-01,2968,472,10311.987,2805432.422,24456.199554,103000.0,-78543.800446,103000.0,2223.290869,2805.432422,-321.161104


In [19]:
# ==========================
# 1. KPIs de viagens
# ==========================
viagens_validas = (
    df_viagens.loc[
        df_viagens[
            "ocupacao_estado"
        ].eq("OK")
    ]
    .copy()
)

kpi_viagens = pd.DataFrame({
    "metrica": [
        "Viagens",
        "Paletes",
        "Peso ton",
        "Ocupação média %",
        "Custo por viagem",
        "Custo por palete",
        "Custo por ton",
        "Margem",
    ],
    "valor": [
        df_viagens["CODEUT"].nunique(),
        df_viagens["paletes"].sum(),
        df_viagens["peso_ton"].sum(),
        viagens_validas[
            "ocupacao_pct"
        ].mean(),
        df_viagens[
            "custo_total"
        ].mean(),
        dividir_seguro(
            pd.Series([
                df_viagens[
                    "custo_total"
                ].sum()
            ]),
            pd.Series([
                df_viagens[
                    "paletes"
                ].sum()
            ]),
        ).iloc[0],
        dividir_seguro(
            pd.Series([
                df_viagens[
                    "custo_total"
                ].sum()
            ]),
            pd.Series([
                df_viagens[
                    "peso_ton"
                ].sum()
            ]),
        ).iloc[0],
        df_viagens["margem"].sum(),
    ],
})

kpi_viagens

,metrica,valor
0,Viagens,16156.000000
1,Paletes,381072.761000
2,Peso ton,96177.160188
3,Ocupação média %,65.915966
4,Custo por viagem,281.174411
5,Custo por palete,11.920699
6,Custo por ton,47.232147
7,Margem,-254657.892771
